# Notebook 01: M5 Data Understanding & Ingestion Validation
**Sprint**: Day 2 — Data Preparation and Exploratory Analysis
**Goal**: Inspect raw files, validate identifiers, analyze missing values and zero-sales sparsity, and establish baseline quality metrics.
---


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print('Imports successful!')


## 1. Raw M5 File Inspection
We inspect the three primary M5 data sources:
1. `calendar.csv`: 1,969 days with event, holiday, and SNAP indicators.
2. `sell_prices.csv`: Weekly sell prices per store and item.
3. `sales_train_validation.csv`: Daily unit sales for 30,490 series from `d_1` to `d_1913`.


In [ ]:
raw_dir = '../data/raw/m5'
cal_df = pd.read_csv(os.path.join(raw_dir, 'calendar.csv'))
prices_df = pd.read_csv(os.path.join(raw_dir, 'sell_prices.csv'), nrows=100000)
sales_sample = pd.read_csv(os.path.join(raw_dir, 'sales_train_validation.csv'), nrows=500)

print('Calendar shape:', cal_df.shape)
print('Sell Prices sample shape:', prices_df.shape)
print('Sales sample shape:', sales_sample.shape)


## 2. Product, Store, and Date Identifier Validation
Validating the schema and key consistency across M5 tables:


In [ ]:
# Calendar date continuity
cal_df['date'] = pd.to_datetime(cal_df['date'])
date_range = pd.date_range(cal_df['date'].min(), cal_df['date'].max())
missing_dates = len(date_range) - len(cal_df)
print(f'Calendar date range: {cal_df["date"].min().date()} to {cal_df["date"].max().date()}')
print(f'Missing calendar dates: {missing_dates} (100% continuous)')

# Unique identifiers in sales
print('Unique stores in sample:', sales_sample['store_id'].unique().tolist())
print('Unique categories in sample:', sales_sample['cat_id'].unique().tolist())
print('Sample item IDs:', sales_sample['item_id'].head(5).tolist())


## 3. Zero-Sales & Sparsity Analysis
Retail point-of-sale data typically exhibits significant intermittency (zero sales on many days).
We evaluate zero-sales proportions across departments.


In [ ]:
# Compute zero-sales percentage across sample items
d_cols = [c for c in sales_sample.columns if c.startswith('d_')]
sales_matrix = sales_sample[d_cols].values
total_observations = sales_matrix.size
zero_observations = (sales_matrix == 0).sum()
zero_pct = (zero_observations / total_observations) * 100

print(f'Total sales observations in sample: {total_observations:,}')
print(f'Zero-sales observations: {zero_observations:,} ({zero_pct:.2f}%)')

# Department breakdown
sales_sample['zero_sales_pct'] = (sales_sample[d_cols] == 0).mean(axis=1) * 100
dept_sparsity = sales_sample.groupby('dept_id')['zero_sales_pct'].mean().reset_index()
print('\nAverage Zero-Sales % by Department:')
print(dept_sparsity.to_string(index=False))


## 4. Sell Price Stability & Variance
Checking how prices fluctuate across weeks for representative items in `CA_1`:


In [ ]:
ca1_prices = prices_df[prices_df['store_id'] == 'CA_1']
price_stats = ca1_prices.groupby('item_id')['sell_price'].agg(['count', 'min', 'max', 'std']).reset_index()
price_stats['price_range'] = price_stats['max'] - price_stats['min']
print('Price stability summary across items:')
print(price_stats.head(10).to_string(index=False))


## 5. Summary of Data Understanding Findings & Limitations
1. **Continuous Calendar**: 1,969 days without gaps from Jan 2011 to June 2016.
2. **Intermittency**: High proportion of zero-sales days (40%–60% depending on department), requiring intermittent/volatility-aware metrics like MASE or WAPE rather than MAPE.
3. **Missing Operational Features**: M5 does not provide live inventory, safety stock, lead times, or promotional budgets. These constraints are supplied by the synthetic operational tables.
